In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def token_report(text):
    toks = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(toks)
    print(f"Texto: {text}")
    print(f"Tokens ({len(toks)}): {toks}")
    print(f"IDs (primeros 12): {ids[:12]}")
    print("-" * 60)

## Parte 1 - Tokenización
Los LLM y transformers necesitan realizar un proceso de "tokenización". Dividen el texto en subpalabras/tokens, el contexto y el peso de cada token se calculan después en el modelo.

In [ ]:
demo_texts = [
    "Programación",
    "Programacion",
    "The cake is a lie",
    "Hay que mover el cacharro",
    "Dónde caemos gente?"
]
for t in demo_texts:
    token_report(t)

Mientras más tokens, más coste y más longitud de secuencia. Cuesta más saber completamente el contexto.
Muchas veces el tokenizer no necesita saber las palabras completas, con conocer la raíz le sirve para saber el contexto:
- habl - o
- habl - aré
- habl - ador

Esto es muy útil en idiomas con palabras extremadamente largas, ya que permite conocer cada una de las raices de las palabras que lo componen

In [ ]:
demo_texts = [
    "Donaudampfschifffahrtsgesellschaftskapitän",
    "Muvaffakiyetsezlestiricilestiriveremeyebileceklerimizdenmissinizcesine",
    "Pneumonoultramicroscopicsilicovolcanoconiosis"
]
for t in demo_texts:
    token_report(t)

## Ejercicio 1 - Demo con palabras largas
Ver cómo se dividen las siguientes palabras en subpartes frecuentes:
- Electroencefalografista
- Otorrinolaringólogo
- Anticonstitucionalmente

In [ ]:
tarea1_texts = [
    "Electroencefalografista",
    "Otorrinolaringólogo",
    "Anticonstitucionalmente",
]
for t in tarea1_texts:
    token_report(t)

## Ejercicio 2 - Diferencias en el idioma
Algunos tokenizers pueden funcionar en varios idiomas, pero otros pueden tener fallos al ver las diferentes relaciones y calcular la atención.

Eficiencia en diferentes idiomas, ¿cuál es más eficiente?

In [ ]:
tarea2_texts = {
    "es": "Necesito dinero del banco",
    "en": "I need money from the bank",
    "tr": "Bankadan para cekmem gerekiyor",
    "de": "Ich brauche Geld von der Bank",
}

results = []
for lang, text in tarea2_texts.items():
    toks = tokenizer.tokenize(text)
    results.append((lang, text, len(toks), toks))

for lang, text, n, toks in results:
    print(f"{lang} | {n} tokens | {text}")
    print(toks)
    print("-" * 60)

## Ejercicio 3 - Busca y Compara 3 Tokenizers

Cada tokenizer funciona diferente según el idioma, arquitectura y datos de entrenamiento. 
**Busca 3 tokenizers que tu elijas** y compararlos.

👉 **[Hugging Face Hub - Feature Extraction Models](https://huggingface.co/models?pipeline_tag=feature-extraction)**  


### Buscar 3 modelos con estos criterios:

**Modelo optimizado para IDIOMA (no inglés):**
- Busca: `spanish`, `french`, `japanese`, `german`, `portuguese`, etc.

**Modelo GENERATIVO (GPT, LLaMA, Falcon, etc.):**
- Busca: modelos que digan "generative" o nombres como `gpt2`, `mistral`, `llama`

**Modelo LIGHTWEIGHT o FAST:**
- Busca: palabras como `distilled`, `fast`, `small`, `lite`

**Para encontrar el nombre del tokenizer**:
- Lee la tarjeta del modelo (haz clic en el nombre)
- Busca el nombre completo: puede ser `usuario/nombre` o solo `nombre-modelo` (si es oficial)
- Si falla, prueba otro en la categoría

In [ ]:
from transformers import AutoTokenizer
import pandas as pd

tarea3_tests = [
    "sudo rm -rf / --no-preserve-root",  #Comando de linux
    "https://store.steampowered.com/app/753640/Outer_Wilds/",  #URL
    "なぜそれを翻訳するのですか 💀💀💀",  #Emojis y texto
    "#skillissue",  #Hashtags
    "AAAAAAA BBBBBB CCCCCC",  #Palabras largas repetidas
    "SELECT * FROM Users WHERE UserId = 105 OR 1=1;", #SQL
    "function test() { return x * 2; }", #Código
    "𓀀𓀁𓀂𓀃𓀄" #Caracteres extraños
]

# TODO: Reemplaza estos con tus modelos buscados
model_language =          #Cambiar: tu modelo para idioma
model_generative =        #Cambiar: tu modelo generativo
model_lightweight =       #Cambiar: tu modelo lightweight

my_models = {
    "Idioma": model_language,
    "Generativo": model_generative,
    "Lightweight": model_lightweight,
}

all_results = {}
for category, model_name in my_models.items():
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        results = []
        for test_text in tarea3_tests:
            toks = tokenizer.tokenize(test_text)
            results.append((len(toks), test_text, toks))
        
        all_results[category] = (model_name, results)
    except Exception as e:
        print(f"El modelo {model_name} falló al cargar.")

#Análisis detallado de los resultados
rows = []

for category, model_name in my_models.items():
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        for text in tarea3_tests:
            tokens = tokenizer.tokenize(text)
            
            rows.append({
                "Modelo": category,
                "Texto": text,
                "Nº tokens": len(tokens),
                "Tokens": " ".join(tokens[:15]) + (" ..." if len(tokens) > 15 else "")
            })
    
    except Exception:
        rows.append({
            "Modelo": category,
            "Texto": "ERROR AL CARGAR MODELO",
            "Nº tokens": "-",
            "Tokens": "-"
        })

df = pd.DataFrame(rows)

pivot_df = df.pivot(index="Texto", columns="Modelo", values=["Nº tokens", "Tokens"])
b
styled_df = pivot_df.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]},
])

display(styled_df)

## Embeddings: ¿Cómo entienden los modelos el significado?

Después de **tokenizar** (dividir en piezas), el modelo convierte cada token en un **número (o vector)**. Esto se llama **embedding**.

**Idea principal**: Palabras con significado similar tienen embeddings parecidos.

**Similitud**: la medimos con **cosine similarity** (número entre -1 y 1; cuanto más cerca a 1, más parecidas):
- "banco" (dinero) ↔ "cuenta bancaria" = similitud alta (~0.8) 
- "banco" (dinero) ↔ "banco" (mueble) = similitud media (~0.4) 
- "banco" (dinero) ↔ "pizza" = similitud baja (~0.1) 

### Alta similitud ≠ Mismo significado

"Me encanta el chocolate" y "Odio el chocolate" tienen similitud muy alta (~0.88), pero significado **completamente opuesto**.

¿Por qué? Comparten estructura idéntica pero el transformer a veces prioriza el contexto temático sobre el sentimiento.

### Tipos de palabras y su impacto en significado

No todas las palabras tienen igual importancia. Cambiar unas genera más cambio de significado que otras:

| Tipo | Impacto | Ejemplos |
|------|---------|----------|
| **Verbos (acción)** | ⬛⬛⬛ ALTO | cerrar->abrir, tirar->recoger, amar->odiar |
| **Sustantivos (entidad)** | ⬛⬛⬛ ALTO | gato->perro, restaurante->hospital, comida->veneno |
| **Adjetivos (cualidad)** | ⬛⬛ MEDIO | grande->pequeño, blanco->negro |
| **Preposiciones/Artículos** | ⬛ BAJO | de->con, el->un (cambio mínimo) |

## Ejercicio 4 - Análisis de Similitud Semántica

**Parte 1: Encuentra pares con similitud alta (>0.8) pero significado diferente**

Busca o construye 2-3 pares de frases donde el modelo calcula similitud alta (>0.8) pero el significado sea *completamente diferente*. Esto muestra los límites y confusiones del modelo.

**Parte 2: Cambia una palabra y observa cómo cambia la similitud**

Se te proporciona una frase base. Modifica **UNA SOLA PALABRA** para cambiar drásticamente su similitud con una frase objetivo. 
Ejemplo: "El banco cerró temprano" -> cambiar "banco" por otra palabra, o "cerró" por otro verbo.

Observa cómo el transformer es sensible al contexto.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

# Modelo base
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)


def sentence_embedding(text):
    """
    Convierte una frase en un embedding (vector numérico)
    usando el promedio de los embeddings de sus tokens.
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    return outputs.last_hidden_state.mean(dim=1)


# ==========================================================
# PARTE 1: Alta similitud, distinto significado
# ==========================================================

print("=" * 60)
print("PARTE 1: Pares con alta similitud pero significado diferente")
print("=" * 60)

print("\nDefine al menos dos pares de frases que:")
print("- tengan alta similitud (≈ > 0.8)")
print("- pero no signifiquen lo mismo\n")

# Lista de tuplas (frase1, frase2)
part1_pairs = [
    ("El gato está en el tejado", "El perro está en el tejado"),
    ("Adoro el chocolate", "Detesto el chocolate"),
    # Añade aquí tus propios ejemplos
]

results_part1 = []

for t1, t2 in part1_pairs:
    emb1 = sentence_embedding(t1)
    emb2 = sentence_embedding(t2)
    sim = F.cosine_similarity(emb1, emb2).item()
    
    results_part1.append((sim, t1, t2))

# Ordenar resultados
results_part1.sort(key=lambda x: x[0], reverse=True)

print("\nResultados:\n")
for sim, t1, t2 in results_part1:
    print(f"{sim:.3f}")
    print(f"  {t1}")
    print(f"  {t2}\n")

In [ ]:
# ==========================================================
# PARTE 2: Sensibilidad a cambios mínimos
# ==========================================================

base_phrase = "El banco cerró temprano"

print(f"\nFrase base:      {base_phrase}")

#Modifica UNA sola palabra para alterar la similitud.
modified_phrase = None      #TODO: Sustituir por la frase modificada (cambiar solo una palabra)
target_phrase = "El banco abre de 9 a 17"  # Frase de comparación

print(f"Frase modificada: {modified_phrase}")
print(f"Frase objetivo:   {target_phrase}\n")

if modified_phrase != base_phrase:
    emb_base = sentence_embedding(base_phrase)
    emb_modified = sentence_embedding(modified_phrase)
    emb_target = sentence_embedding(target_phrase)

    sim_base = F.cosine_similarity(emb_base, emb_target).item()
    sim_modified = F.cosine_similarity(emb_modified, emb_target).item()

    print(f"Similitud (base vs objetivo):      {sim_base:.3f}")
    print(f"Similitud (modificada vs objetivo): {sim_modified:.3f}")
    print(f"Diferencia:                        {sim_modified - sim_base:+.3f}")
else:
    print("Debes modificar una palabra en 'modified_phrase'.")

## Análisis

Qué puede estar influyendo en la similitud? ¿Depende del tipo de palabra modificada?